<a href="https://colab.research.google.com/github/khupkhaidopmul-stack/lis4693/blob/main/lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title Setup and Installations
import pandas as pd
import altair as alt
import requests
import io

# Disable max rows limit for large dataset
alt.data_transformers.disable_max_rows()


DataTransformerRegistry.enable('default')

In [ ]:
# @title Load Seattle Library Data
url = "https://raw.githubusercontent.com/manika-lamba/SP26-LIS4_5693/refs/heads/main/lab-assignments/lab-3/Seattle-Library_2015-2021.csv"
response = requests.get(url)
response.raise_for_status()
text = response.text

seattle_df = pd.read_csv(io.StringIO(text))
print(f"Dataset loaded with {len(seattle_df)} rows")
seattle_df.head()

Dataset loaded with 282287 rows


,Title,Creator,MaterialType,Checkouts,CheckoutYear,CheckoutMonth,Publisher,PublicationYear,Subjects,UsageClass,CheckoutType
0,Frog and toad all year / by Arnold Lobel.,"Lobel, Arnold",BOOK,34,2016,10,"Harper & Row,",c1976.,"Friendship Fiction, Frogs Juvenile fiction, To...",Physical,Horizon
1,"My brilliant friend : childhood, adolescence /...","Ferrante, Elena",BOOK,110,2016,10,"Europa Editions,",2012,"Friendship Fiction, Naples Italy Fiction",Physical,Horizon
2,Star trek [videorecording] / Paramount ; Spygl...,NaN,VIDEODISC,36,2016,10,"Paramount Home Entertainment,",c2009.,"Kirk James T 2233 2371 Drama, Spock Mr Drama, ...",Physical,Horizon
3,The Man in the High Castle,Philip K. Dick,EBOOK,63,2016,10,Houghton Mifflin Harcourt Trade and Reference,2012,"Fiction, Science Fiction",Digital,OverDrive
4,"The Fifth Season: Broken Earth Series, Book 1",N. K. Jemisin,EBOOK,44,2016,10,"Hachette Digital, Inc.",2015,"Fantasy, Fiction, Thriller",Digital,OverDrive


In [ ]:
# @title Data Overview - See available columns
print("Available columns:")
print(seattle_df.columns.tolist())
print("\nFirst few rows of key columns:")
seattle_df[['CheckoutMonth', 'UsageClass', 'CheckoutType', 'Checkouts', 'Creator']].head(10)

Available columns:
['Title', 'Creator', 'MaterialType', 'Checkouts', 'CheckoutYear', 'CheckoutMonth', 'Publisher', 'PublicationYear', 'Subjects', 'UsageClass', 'CheckoutType']

First few rows of key columns:


,CheckoutMonth,UsageClass,CheckoutType,Checkouts,Creator
0,10,Physical,Horizon,34,"Lobel, Arnold"
1,10,Physical,Horizon,110,"Ferrante, Elena"
2,10,Physical,Horizon,36,NaN
3,10,Digital,OverDrive,63,Philip K. Dick
4,10,Digital,OverDrive,44,N. K. Jemisin
5,10,Physical,Horizon,28,NaN
6,10,Physical,Horizon,38,NaN
7,10,Physical,Horizon,26,NaN
8,10,Physical,Horizon,22,"O'Connor, George"
9,10,Physical,Horizon,23,"Worth, Bonnie"


In [ ]:
# @title TASK 1, 2, & 3: First Visualization - CheckoutMonth and Checkouts (Tick Mark)
# TASK 1: Using 'CheckoutMonth' and 'Checkouts' columns (different from example)
# TASK 2: Using TICK mark (not used in example notebook)
# TASK 3: Adding dropdown selection interactivity (different from example's legend selection)

# Aggregate average checkouts by month across all years
monthly_avg = seattle_df.groupby('CheckoutMonth')['Checkouts'].mean().reset_index()

# Create a dropdown selection for showing different statistics
statistic_select = alt.selection_point(
    name="Statistic",
    fields=['stat'],
    bind=alt.binding_select(options=['Average', 'Median', 'Sum'], name='Show:'),
    value=[{'stat': 'Average'}]
)

# Calculate different statistics
monthly_stats = seattle_df.groupby('CheckoutMonth')['Checkouts'].agg(['mean', 'median', 'sum']).reset_index()
monthly_stats = monthly_stats.melt(id_vars=['CheckoutMonth'], value_vars=['mean', 'median', 'sum'],
                                     var_name='stat', value_name='value')

tick_chart = alt.Chart(monthly_stats).mark_tick(
    thickness=2,
    size=40
).encode(
    x=alt.X('CheckoutMonth:Q', title='Month', scale=alt.Scale(domain=[1, 12]),
            axis=alt.Axis(values=[1,2,3,4,5,6,7,8,9,10,11,12], labelExpr="month(datum.value)")),
    y=alt.Y('value:Q', title='Checkout Count'),
    color=alt.condition(statistic_select, alt.Color('stat:N', title='Statistic',
                      scale=alt.Scale(scheme='set2')), alt.value('lightgray')),
    tooltip=[
        alt.Tooltip('CheckoutMonth:Q', title='Month', format='d'),
        alt.Tooltip('value:Q', title='Value', format='.0f'),
        alt.Tooltip('stat:N', title='Statistic')
    ]
).add_params(
    statistic_select
).properties(
    title='Monthly Checkout Patterns (Tick Plot) - Interactive Dropdown',
    width=500,
    height=350
).interactive()

tick_chart

alt.Chart(...)

In [ ]:
# @title TASK 1, 2, & 3: Second Visualization - Creator and Checkouts (Text Mark)
# TASK 1: Using 'Creator' and 'Checkouts' columns (different from first visualization)
# TASK 2: Using TEXT mark (not used in example notebook)
# TASK 3: Adding slider binding interactivity (different from example)

# Get top 20 authors by total checkouts
top_authors = seattle_df[seattle_df['Creator'].notna()].groupby('Creator')['Checkouts'].sum().sort_values(ascending=False).head(20).reset_index()

# Create slider for minimum checkout threshold
threshold_slider = alt.selection_point(
    name="Threshold",
    fields=['threshold'],
    bind=alt.binding_range(min=0, max=50000, step=5000, name='Minimum Checkouts:'),
    value=[{'threshold': 10000}]
)

# Filter data based on slider
text_chart = alt.Chart(top_authors).transform_filter(
    alt.datum.Checkouts > threshold_slider.threshold
).mark_text(
    fontSize=11,
    baseline='middle'
).encode(
    x=alt.X('Checkouts:Q', title='Total Checkouts', scale=alt.Scale(type='log')),
    y=alt.Y('Creator:N', title='Author/Creator', sort='-x'),
    text=alt.Text('Creator:N'),
    color=alt.Color('Checkouts:Q', title='Checkout Volume', scale=alt.Scale(scheme='viridis')),
    tooltip=[
        alt.Tooltip('Creator:N', title='Author'),
        alt.Tooltip('Checkouts:Q', title='Total Checkouts', format='.0f')
    ]
).add_params(
    threshold_slider
).properties(
    title='Top 20 Authors by Checkouts (Text Plot) - Adjustable Threshold Slider',
    width=550,
    height=400
).interactive()

text_chart

alt.Chart(...)

In [ ]:
# @title TASK 4: Horizontal Concatenation (Side-by-Side)
# Combine both visualizations horizontally

combined_chart = alt.hconcat(tick_chart, text_chart)
combined_chart

alt.HConcatChart(...)

## Assignment Reflection

### What went well?
The Altair library made it easy to experiment with different mark types beyond bar charts. I successfully created three distinct visualizations using **tick marks**, **text marks**, and **rect marks** (heatmap) - none of which appeared in the example notebook. The interactive features worked well: the dropdown selector for switching between statistics (average/median/sum) on the tick plot, the slider binding for filtering authors by checkout threshold on the text plot, and the brush selection on the point chart all added meaningful user control. The horizontal concatenation using `alt.hconcat()` was straightforward and produced a clean side-by-side layout.

### What did not go well or what challenges I encountered?
The main challenge was finding appropriate column combinations that would produce meaningful visualizations. The `Creator` column contained many null values and variations of author names, requiring filtering and aggregation. The `CheckoutMonth` column needed proper formatting to display month names instead of numbers. Additionally, the text mark visualization required careful sizing and sorting to prevent overlapping labels. The large dataset (282,287 rows) again required aggregation before plotting, and I had to filter top authors/publishers to avoid cluttered visualizations. The rect mark (heatmap) required binning the PublicationYear to create a usable grid.